# 01 — Extração: SQL Server → MinIO `landing-zone`

Lê todas as tabelas do SQL Server e salva cada uma como arquivo **CSV** no bucket `landing-zone`.

| Origem | Destino |
|---|---|
| `seguradora.<tabela>` | `landing-zone/<tabela>/<tabela>.csv` |

In [ ]:
import io
import os
import csv
import pyodbc
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

## Conexões

In [ ]:
conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={os.getenv('SQLSERVER_HOST')},{os.getenv('SQLSERVER_PORT')};"
    f"DATABASE={os.getenv('SQLSERVER_DB')};"
    f"UID={os.getenv('SQLSERVER_USER')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

conn   = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("SQL Server:", os.getenv("SQLSERVER_DB"))

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY"),
    config=Config(signature_version="s3v4"),
)
LANDING = os.getenv("MINIO_LANDING_BUCKET")
print("MinIO bucket:", LANDING)

## Criação do bucket `landing-zone`

In [ ]:
existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if LANDING not in existing:
    s3.create_bucket(Bucket=LANDING)
    print(f"Bucket '{LANDING}' criado.")
else:
    print(f"Bucket '{LANDING}' já existe.")

## Extração e upload

In [ ]:
# Lista todas as tabelas do banco
cursor.execute(
    "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES "
    "WHERE TABLE_TYPE = 'BASE TABLE' ORDER BY TABLE_NAME"
)
TABLES = [row[0] for row in cursor.fetchall()]
print("Tabelas encontradas:", TABLES)

for table in TABLES:
    cursor.execute(f"SELECT * FROM {table}")
    rows   = cursor.fetchall()
    cols   = [desc[0] for desc in cursor.description]

    # Serializa para CSV em memória
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(cols)
    writer.writerows(rows)
    payload = buf.getvalue().encode("utf-8")

    key = f"{table}/{table}.csv"
    s3.put_object(
        Bucket=LANDING,
        Key=key,
        Body=io.BytesIO(payload),
        ContentType="text/csv",
    )
    print(f"  {table}: {len(rows)} docs → s3://{LANDING}/{key}")

conn.close()
print("\nExtração concluída.")

## Verificação — listar arquivos no `landing-zone`

In [ ]:
response = s3.list_objects_v2(Bucket=LANDING)
print(f'Arquivos em "{LANDING}":')
for obj in response.get("Contents", []):
    size_kb = obj["Size"] / 1024
    print(f'  {obj["Key"]}  ({size_kb:.1f} KB)')